In [2]:
# ============================================================
# 07_phase3_DE_pseudobulk_corrected.ipynb
# Phase 3 — Corrected Pseudobulk DE Analysis
# Uses true raw integer counts via Option 1:
# subset GSE114725_raw.h5ad / GSE176078_raw.h5ad to QC-surviving barcodes
# Supersedes notebook 06 which used log-normalised data
# ============================================================

# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3" / "corrected"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_labels_1 = {
    "0": "T cells",
    "1": "Activated T cells",
    "2": "NK/Cytotoxic T cells",
    "3": "Macrophages",
    "4": "B cells",
    "5": "Monocytes/DC"
}

print("Ready")

Ready


In [3]:
# ----------------------------
# Cell 2 — Load true raw counts (GSE114725)
# Option 1: subset raw h5ad to QC-surviving barcodes AND genes
# Verified: integer counts, max ~1236, np.all == int = True
# ----------------------------

# Load true raw counts
adata_raw = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")

# Load QC-surviving barcodes and genes from annotated object
adata_annotated = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")

# Subset to QC-surviving cells AND genes
adata_raw_qc = adata_raw[
    adata_annotated.obs_names,        # QC-surviving cells
    adata_annotated.raw.var_names     # QC-surviving genes
].copy()

# Transfer cell type and tissue labels
adata_raw_qc.obs["cell_type"] = adata_annotated.obs["cell_type"].values
adata_raw_qc.obs["tissue"] = adata_annotated.obs["tissue"].values
adata_raw_qc.obs["patient"] = adata_annotated.obs["patient"].values

del adata_raw
del adata_annotated
gc.collect()

# Verify integer counts
X_check = adata_raw_qc.X[0:100]
if issparse(X_check): X_check = X_check.toarray()
print(f"Cells: {adata_raw_qc.n_obs}")
print(f"Genes: {adata_raw_qc.n_vars}")
print(f"Max value: {adata_raw_qc.X.max()}")
print(f"Integer counts: {np.all(X_check == X_check.astype(int))}")
print(f"\nCell type counts:")
print(adata_raw_qc.obs["cell_type"].value_counts())
print(f"\nTissue counts:")
print(adata_raw_qc.obs["tissue"].value_counts())

Cells: 44662
Genes: 14800
Max value: 1236.0
Integer counts: True

Cell type counts:
cell_type
T cells                 16554
NK/Cytotoxic T cells     9811
Macrophages              8624
Activated T cells        5353
Monocytes/DC             3530
B cells                   790
Name: count, dtype: int64

Tissue counts:
tissue
TUMOR        19594
BLOOD        15592
LYMPHNODE     5136
NORMAL        4340
Name: count, dtype: int64


In [5]:
# ----------------------------
# Cell 3 — Pseudobulk aggregation function
# ----------------------------
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        counts_list.append(X_sample.sum(axis=0))
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample, condition_col
        ].iloc[0]
        meta_list.append({sample_col: sample, condition_col: condition})
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    return counts_df, meta_df

print("Function defined")

Function defined


In [13]:
# ----------------------------
# Cell 4 — Broad cluster DE (TUMOR vs BLOOD, all cell types)
# n_cpus=1 prevents worker crashes on Windows
# ----------------------------
cell_types_de = [
    "T cells",
    "NK/Cytotoxic T cells",
    "Activated T cells",
    "Macrophages",
    "Monocytes/DC"
]
# B cells excluded — only 790 cells, likely insufficient pseudobulk power

all_results = {}

for ct in cell_types_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata_raw_qc,
            cell_type=ct,
            sample_col="patient",
            condition_col="tissue"
        )

        # Filter to TUMOR vs BLOOD only
        mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
        counts_df = counts_df[mask]
        meta_df = meta_df[mask]

        condition_counts = meta_df["tissue"].value_counts()
        print(f"  Samples per condition: {condition_counts.to_dict()}")
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples")
            continue

        inference = DefaultInference(n_cpus=1)
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~tissue",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["tissue", "TUMOR", "BLOOD"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df

        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")

        print(f"  Significant DEGs: {len(sig_df)}")
        if len(sig_df) > 0:
            print("\n  Top upregulated in TUMOR:")
            print(sig_df[sig_df["log2FoldChange"] > 0].head(5)[
                ["log2FoldChange", "padj"]].to_string())
            print("\n  Top downregulated in TUMOR:")
            print(sig_df[sig_df["log2FoldChange"] < 0].head(5)[
                ["log2FoldChange", "padj"]].to_string())

        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood.csv")
        sig_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood_significant.csv")

        all_results[ct] = {"full": results_df, "sig": sig_df}

    except Exception as e:
        print(f"  Failed: {e}")

print("\nBroad cluster DE complete")


Running DE for: T cells

T cells: 16554 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}
  Samples per condition: {'TUMOR': 3, 'BLOOD': 2}


Fitting size factors...
... done in 0.03 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 33.67 seconds.

Fitting dispersion trend curve...
... done in 1.06 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 31.25 seconds.

Fitting LFCs...
... done in 19.58 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 5.17 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     80.966918       -1.037182  0.524032 -1.979233  0.047790  0.309348
A2M      25.390227        2.826873  0.822349  3.437560  0.000587  0.014045
A4GALT    0.570732        1.466568  3.855582  0.380375  0.703667       NaN
AAAS     33.829982       -0.265937  0.555217 -0.478979  0.631954  0.904820
AACS     17.092119       -0.780216  0.715659 -1.090206  0.275622  0.728056
...            ...             ...       ...       ...       ...       ...
ZXDC     31.081020       -0.605611  0.522995 -1.157967  0.246877  0.699050
ZYG11B   43.678391       -0.455815  0.536409 -0.849753  0.395463  0.807311
ZYX     329.185731        0.067290  0.425381  0.158188  0.874308  0.978233
ZZEF1    79.370439       -0.154199  0.378873 -0.406994  0.684013  0.921154
ZZZ3     31.477569       -0.625898  0.519566 -1.204654  0.228337  0.679454

[14800 rows x 6 columns]
  Significant 

...

Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     80.061365       -1.212431  0.550372 -2.202931  0.027600  0.239676
A2M     455.099314        1.354918  1.020530  1.327660  0.184290  0.585067
A4GALT    3.337959        0.728411  2.170180  0.335646  0.737138       NaN
AAAS     36.016174       -0.138696  0.714864 -0.194017  0.846163  0.965424
AACS     20.480369        0.139860  0.807411  0.173220  0.862478  0.970204
...            ...             ...       ...       ...       ...       ...
ZXDC     43.806203       -0.155008  0.678568 -0.228433  0.819309  0.957787
ZYG11B   71.857280        0.281952  0.644427  0.437523  0.661732  0.909967
ZYX     664.191730        0.808236  0.530115  1.524642  0.127348  0.503814
ZZEF1    93.065197        0.069252  0.515971  0.134217  0.893231  0.976426
ZZZ3     43.352043       -0.698783  0.604169 -1.156603  0.247435  0.666073

[14800 rows x 6 columns]
  Significant 

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 32.65 seconds.

Fitting dispersion trend curve...
... done in 1.07 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 35.34 seconds.

Fitting LFCs...
... done in 23.82 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 5.50 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     52.997465       -1.176197  0.598169 -1.966329  0.049261  0.414259
A2M     764.373696        0.592717  0.761438  0.778418  0.436323  0.873081
A4GALT    2.773192       -1.198606  2.030321 -0.590353  0.554954       NaN
AAAS     38.285297       -0.147152  0.643279 -0.228753  0.819061  0.972808
AACS     17.019225       -1.089299  0.860752 -1.265520  0.205685  0.728839
...            ...             ...       ...       ...       ...       ...
ZXDC     39.670615       -0.181550  0.641602 -0.282964  0.777205  0.967076
ZYG11B   86.450729        0.076921  0.549829  0.139901  0.888738  0.982060
ZYX     765.755425        0.323598  0.399888  0.809221  0.418388  0.865465
ZZEF1    68.307674       -0.468360  0.529812 -0.884011  0.376690  0.849422
ZZZ3     34.228111        0.207973  0.732766  0.283820  0.776548  0.967076

[14800 rows x 6 columns]
  Significant 

In [14]:
# ----------------------------
# Cell 5 — Pathway enrichment on corrected DE results
# ----------------------------
print("Running pathway enrichment on corrected DE results...")

for ct, res in all_results.items():
    print(f"\n{'='*40}")
    print(f"Pathways: {ct}")

    full_df = res["full"].dropna(subset=["padj"])
    sig_df = res["sig"]

    background = full_df.index.tolist()
    up_genes = sig_df[sig_df["log2FoldChange"] > 0].index.tolist()
    down_genes = sig_df[sig_df["log2FoldChange"] < 0].index.tolist()

    print(f"  Up: {len(up_genes)}, Down: {len(down_genes)}, "
          f"Background: {len(background)}")

    for direction, gene_list in [("up", up_genes), ("down", down_genes)]:
        if len(gene_list) < 10:
            print(f"  Skipping {direction} - too few genes ({len(gene_list)})")
            continue
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
                background=background,
                outdir=None,
                verbose=False
            )
            sig_paths = enr.results[
                enr.results["Adjusted P-value"] < 0.05].copy()
            print(f"  {direction}: {len(sig_paths)} significant pathways")
            if len(sig_paths) > 0:
                print(sig_paths[["Gene_set", "Term",
                                  "Adjusted P-value"]].head(5).to_string())

            ct_clean = ct.replace("/", "_").replace(" ", "_")
            enr.results.to_csv(
                RESULTS_DIR / f"GSE114725_pathways_{ct_clean}_{direction}.csv",
                index=False
            )
        except Exception as e:
            print(f"  {direction} failed: {e}")

print("\nPathway enrichment complete")

Running pathway enrichment on corrected DE results...

Pathways: T cells
  Up: 455, Down: 237, Background: 11126
  up: 107 significant pathways
               Gene_set                           Term  Adjusted P-value
0  MSigDB_Hallmark_2020  TNF-alpha Signaling via NF-kB      6.408943e-63
1  MSigDB_Hallmark_2020          Inflammatory Response      1.427414e-25
2  MSigDB_Hallmark_2020            Allograft Rejection      1.175148e-23
3  MSigDB_Hallmark_2020      Interferon Gamma Response      2.785460e-19
4  MSigDB_Hallmark_2020           IL-2/STAT5 Signaling      5.570964e-18
  down: 3 significant pathways
                Gene_set                 Term  Adjusted P-value
0   MSigDB_Hallmark_2020       Myc Targets V1      1.501644e-07
26       KEGG_2021_Human             Ribosome      1.039470e-93
27       KEGG_2021_Human  Coronavirus disease      3.196758e-84

Pathways: NK/Cytotoxic T cells
  Up: 214, Down: 119, Background: 9708
  up: 92 significant pathways
               Gene_set       

In [6]:
# ----------------------------
# Cell 6 — Load GSE176078 true raw counts
# ----------------------------
del adata_raw_qc
gc.collect()

# Load true raw counts
adata_raw2 = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")

# Load corrected labels
labels2 = pd.read_csv(
    PROJECT_DIR / "results" / "phase2_clustering_v2" /
    "GSE176078_cluster_annotations_v2_corrected.csv",
    index_col=0
)

# Load annotated object obs only for barcodes and gene names
adata_ann2 = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    backed="r"
)

# Subset raw to QC-surviving cells AND genes
adata_raw2_qc = adata_raw2[
    adata_ann2.obs_names,
    adata_ann2.raw.var_names
].copy()

# Transfer labels
adata_raw2_qc.obs["cell_type"] = labels2["cell_type"].reindex(
    adata_raw2_qc.obs_names).values
adata_raw2_qc.obs["subtype"] = adata_ann2.obs["subtype"].values
adata_raw2_qc.obs["orig.ident"] = adata_ann2.obs["orig.ident"].values

adata_ann2.file.close()
del adata_raw2
gc.collect()

# Verify
X_check = adata_raw2_qc.X[0:100]
if issparse(X_check): X_check = X_check.toarray()
print(f"Cells: {adata_raw2_qc.n_obs}")
print(f"Genes: {adata_raw2_qc.n_vars}")
print(f"Max value: {adata_raw2_qc.X.max()}")
print(f"Integer counts: {np.all(X_check == X_check.astype(int))}")
print(f"\nCell type counts (immune):")
immune = ["T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
          "B cells", "Plasma cells", "Macrophages", "Monocytes/DC", "pDC"]
print(adata_raw2_qc.obs[
    adata_raw2_qc.obs["cell_type"].isin(immune)]["cell_type"].value_counts())
print(f"\nSubtypes: {adata_raw2_qc.obs['subtype'].unique().tolist()}")

Cells: 91425
Genes: 27343
Max value: 13141.0
Integer counts: True

Cell type counts (immune):
cell_type
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
T cells                  5989
B cells                  2791
Plasma cells             2583
NK cells                 2438
pDC                       315
Name: count, dtype: int64

Subtypes: ['HER2+', 'TNBC', 'ER+']


In [7]:
# Cell 3 — Pseudobulk aggregation function
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        counts_list.append(X_sample.sum(axis=0))
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample, condition_col
        ].iloc[0]
        meta_list.append({sample_col: sample, condition_col: condition})
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    print(f"  Pseudobulk: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    return counts_df, meta_df

print("Function defined")

Function defined


In [5]:
# Cell 4 — GSE176078 subtype DE (all three comparisons)
immune_types = [
    "T cells", "CD8 T cells", "Macrophages",
    "NK cells", "B cells", "Naive/memory T cells"
]

comparisons = [
    ("TNBC", "ER+"),
    ("TNBC", "HER2+"),
    ("HER2+", "ER+")
]

all_results2 = {}

for ct in immune_types:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata_raw2_qc,
            cell_type=ct,
            sample_col="orig.ident",
            condition_col="subtype"
        )

        condition_counts = meta_df["subtype"].value_counts()
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples")
            continue

        inference = DefaultInference(n_cpus=1)
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~subtype",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        ct_clean = ct.replace("/", "_").replace(" ", "_")

        for test, ref in comparisons:
            try:
                stat_res = DeseqStats(
                    dds,
                    contrast=["subtype", test, ref],
                    inference=inference
                )
                stat_res.summary()
                results_df = stat_res.results_df

                sig_df = results_df[
                    (results_df["padj"] < 0.05) &
                    (abs(results_df["log2FoldChange"]) > 0.5)
                ].copy().sort_values("padj")

                print(f"  {test} vs {ref}: {len(sig_df)} DEGs")

                results_df.to_csv(
                    RESULTS_DIR /
                    f"GSE176078_DE_{ct_clean}_{test}_vs_{ref}.csv")
                sig_df.to_csv(
                    RESULTS_DIR /
                    f"GSE176078_DE_{ct_clean}_{test}_vs_{ref}_significant.csv")

                all_results2[f"{ct}_{test}_vs_{ref}"] = {
                    "full": results_df, "sig": sig_df}

            except Exception as e:
                print(f"  {test} vs {ref} failed: {e}")

    except Exception as e:
        print(f"  Failed: {e}")

print("\nGSE176078 subtype DE complete")


Running DE for: T cells

T cells: 5989 cells
  Pseudobulk: (25, 27343)
  Conditions: {'TNBC': 10, 'ER+': 10, 'HER2+': 5}


Fitting size factors...
... done in 0.09 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 45.75 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 1.51 seconds.

Fitting MAP dispersions...
... done in 66.49 seconds.

Fitting LFCs...
... done in 87.48 seconds.

Calculating cook's distance...
... done in 0.20 seconds.

Replacing 109 outlier genes.

Fitting dispersions...
... done in 0.29 seconds.

Fitting MAP dispersions...
... done in 0.34 seconds.

Fitting LFCs...
... done in 0.45 seconds.

Running Wald tests...
... done in 11.03 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.239172        0.632797  3.581127  0.176703  0.859741   
FO538757.3     0.000000             NaN       NaN       NaN       NaN   
FO538757.2     5.933307       -0.543132  0.548539 -0.990141  0.322105   
AP006222.2     0.698842       -0.149340  0.836777 -0.178470  0.858354   
RP4-669L17.10  0.017363        0.375512  2.733367  0.137381  0.890730   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.999892  
FO538757.3   

...

Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.058048       -0.872301  1.688859 -0.516503  0.605503   
FO538757.3     0.031656       -1.122598  2.988904 -0.375588  0.707223   
FO538757.2     8.534872       -0.113867  0.461583 -0.246689  0.805149   
AP006222.2     1.787547       -0.272351  0.848198 -0.321094  0.748139   
RP4-669L17.10  0.129547       -0.608551  1.479477 -0.411328  0.680832   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.984701  
FO538757.3   

Running Wald tests...
... done in 13.83 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs HER2+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.058048        2.585323  2.276306  1.135754  0.256060   
FO538757.3     0.031656        2.311915  3.732024  0.619480  0.535600   
FO538757.2     8.534872        0.088768  0.507263  0.174993  0.861085   
AP006222.2     1.787547       -0.149961  0.916705 -0.163587  0.870057   
RP4-669L17.10  0.129547        1.019228  1.661351  0.613493  0.539550   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.956141  
FO538757.3 

Running Wald tests...
... done in 12.09 seconds.



Log2 fold change & Wald test p-value: subtype HER2+ vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.058048       -3.457623  2.247616 -1.538351  0.123963   
FO538757.3     0.031656       -3.434513  3.601936 -0.953519  0.340327   
FO538757.2     8.534872       -0.202635  0.505509 -0.400853  0.688528   
AP006222.2     1.787547       -0.122391  0.914222 -0.133874  0.893502   
RP4-669L17.10  0.129547       -1.627779  1.661903 -0.979467  0.327349   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.890397  
FO538757.3  

In [9]:
# ----------------------------
# Cell 5 — GSE176078 ER+ vs TNBC (better powered, exclude HER2+)
# ----------------------------

immune_types = [
    "T cells", "CD8 T cells", "Macrophages",
    "NK cells", "B cells", "Naive/memory T cells"
]

print("Running ER+ vs TNBC (excluding HER2+)...")

adata_ertnbc = adata_raw2_qc[
    adata_raw2_qc.obs["subtype"].isin(["ER+", "TNBC"])].copy()

all_results_ertnbc = {}

for ct in immune_types:
    print(f"\n{'='*40}")
    print(f"{ct}")

    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata_ertnbc,
            cell_type=ct,
            sample_col="orig.ident",
            condition_col="subtype"
        )

        condition_counts = meta_df["subtype"].value_counts()
        print(f"  Samples per subtype: {condition_counts.to_dict()}")
        if condition_counts.min() < 2:
            print(f"  Skipping")
            continue

        inference = DefaultInference(n_cpus=1)
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~subtype",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["subtype", "TNBC", "ER+"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df

        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")

        print(f"  TNBC vs ER+: {len(sig_df)} DEGs")
        if len(sig_df) > 0:
            print("\n  Top upregulated in TNBC:")
            print(sig_df[sig_df["log2FoldChange"] > 0].head(5)[
                ["log2FoldChange", "padj"]].to_string())
            print("\n  Top downregulated in TNBC:")
            print(sig_df[sig_df["log2FoldChange"] < 0].head(5)[
                ["log2FoldChange", "padj"]].to_string())

        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(
            RESULTS_DIR / f"GSE176078_DE_{ct_clean}_TNBC_vs_ERplus.csv")
        sig_df.to_csv(
            RESULTS_DIR / f"GSE176078_DE_{ct_clean}_TNBC_vs_ERplus_significant.csv")

        all_results_ertnbc[ct] = {"full": results_df, "sig": sig_df}

    except Exception as e:
        print(f"  Failed: {e}")

print("\nER+ vs TNBC DE complete")

Running ER+ vs TNBC (excluding HER2+)...

T cells

T cells: 4008 cells
  Pseudobulk: (20, 27343)
  Conditions: {'TNBC': 10, 'ER+': 10}
  Samples per subtype: {'TNBC': 10, 'ER+': 10}


Fitting size factors...
... done in 0.09 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 29.52 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 1.36 seconds.

Fitting MAP dispersions...
... done in 53.47 seconds.

Fitting LFCs...
... done in 77.18 seconds.

Calculating cook's distance...
... done in 0.11 seconds.

Replacing 71 outlier genes.

Fitting dispersions...
... done in 0.13 seconds.

Fitting MAP dispersions...
... done in 0.26 seconds.

Fitting LFCs...
... done in 0.29 seconds.

Running Wald tests...
... done in 12.31 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.208285        0.619367  3.257627  0.190128  0.849209   
FO538757.3     0.000000             NaN       NaN       NaN       NaN   
FO538757.2     4.332616       -0.564489  0.603711 -0.935031  0.349772   
AP006222.2     0.325309       -0.132330  0.855825 -0.154623  0.877119   
RP4-669L17.10  0.015370        0.337381  2.188051  0.154193  0.877458   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.999966  
FO538757.3   

...

Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  padj
RP11-34P13.7   0.017970       -3.354146  2.233006 -1.502077  0.133077   NaN
FO538757.3     0.014763       -2.388262  2.184239 -1.093407  0.274215   NaN
FO538757.2     1.155243       -0.379805  0.835579 -0.454542  0.649439   NaN
AP006222.2     0.295603       -0.363996  1.045753 -0.348071  0.727787   NaN
RP4-669L17.10  0.023196       -2.219850  1.743209 -1.273428  0.202866   NaN
...                 ...             ...       ...       ...       ...   ...
MTNR1B         0.000000             NaN       NaN       NaN       NaN   NaN
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   NaN
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   NaN
LINC01570      0.000000             NaN       NaN       NaN       NaN   NaN
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   NaN

[27343 rows x 6 columns]
  TN

Fitting size factors...
... done in 0.07 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 40.05 seconds.

Fitting dispersion trend curve...
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 1.28 seconds.

Fitting MAP dispersions...
... done in 62.98 seconds.

Fitting LFCs...
... done in 59.99 seconds.

Calculating cook's distance...
... done in 0.10 seconds.

Replacing 135 outlier genes.

Fitting dispersions...
... done in 0.27 seconds.

Fitting MAP dispersions...
... done in 0.36 seconds.

Fitting LFCs...
... done in 0.44 seconds.

Running Wald tests...
... done in 15.24 seconds.



Log2 fold change & Wald test p-value: subtype TNBC vs ER+
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RP11-34P13.7   0.052084       -0.919870  1.388713 -0.662390  0.507721   
FO538757.3     0.028436       -1.147017  2.342234 -0.489711  0.624339   
FO538757.2     6.206867       -0.120876  0.502878 -0.240368  0.810045   
AP006222.2     1.206419       -0.258004  0.882300 -0.292422  0.769964   
RP4-669L17.10  0.082104       -0.619143  1.437721 -0.430642  0.666729   
...                 ...             ...       ...       ...       ...   
MTNR1B         0.000000             NaN       NaN       NaN       NaN   
RP11-95I16.2   0.000000             NaN       NaN       NaN       NaN   
RP11-383C5.8   0.000000             NaN       NaN       NaN       NaN   
LINC01570      0.000000             NaN       NaN       NaN       NaN   
RP4-668E10.4   0.000000             NaN       NaN       NaN       NaN   

                   padj  
RP11-34P13.7   0.992598  
FO538757.3   